Libraries

In [286]:
import numpy as np
import os
import random
from PIL import Image, ImageEnhance, ImageFilter, ImageOps, ImageStat, ImageDraw
import cv2 # Wymagane dla transformacji perspektywy
import time # Do mierzenia czasu

Loading and preparing files

In [287]:
PANEL_DIR = "panele"     # Katalog z panelami PNG (przezroczyste tło)
ROOF_DIR = "dachy"           # Katalog ze zdjęciami dachów (tła)
OUTPUT_BASE_DIR = "aug"

AUG_PER_COMPOSITE = 50           # Ilość generowanych obrazów na każdy plik panelu
CLASS_ID = 0                     # Indeks klasy YOLO (Panel fotowoltaiczny)

# --- NOWA ZMIENNA: Normalizacja Tła ---
# Wszystkie tła zostaną przeskalowane do tego rozmiaru
BACKGROUND_TARGET_SIZE = (1024, 1024) 

# --- USTAWIENIA REALIZMU ---
PERSPECTIVE_CHANCE = 0.9  # 90% szans na zniekształcenie perspektywy
PERSPECTIVE_STRENGTH = (0.1, 0.25) # Min/Max siła zniekształcenia

SCALE_RANGE = (0.15, 0.50) # Min/Max % szerokości tła, jaki zajmie panel
POSITION_MAX_Y = 0.60 # Gwarancja umieszczenia na dachu (górne 60% obrazu)

# --- NOWA ZMIENNA: Minimalny rozmiar panelu ---
MIN_PIXEL_SIZE = 30   # Minimalny rozmiar panelu (10x10 px)

# --- Tworzenie katalogów ---
OUTPUT_IMG_DIR = os.path.join(OUTPUT_BASE_DIR, "obrazy")
OUTPUT_LABEL_DIR = os.path.join(OUTPUT_BASE_DIR, "etykiety")

os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_LABEL_DIR, exist_ok=True)

print(f"Katalogi wyjściowe gotowe w: {OUTPUT_BASE_DIR}")
print(f"Tła będą normalizowane do: {BACKGROUND_TARGET_SIZE}")
print(f"Minimalny rozmiar panelu: {MIN_PIXEL_SIZE}x{MIN_PIXEL_SIZE} px")

Katalogi wyjściowe gotowe w: aug
Tła będą normalizowane do: (1024, 1024)
Minimalny rozmiar panelu: 30x30 px


Preztwarzanie samego panela - modyfikacje

In [288]:
def apply_scaling(img, abs_boxes):
    width, height = img.size
    # Sprawdzenie, czy wymiary są poprawne przed skalowaniem
    if width <= 0 or height <= 0:
        return img, abs_boxes # Nie można skalować obrazu o zerowym wymiarze
    scale = random.uniform(0.7, 1.3)
    new_w = int(width * scale)
    new_h = int(height * scale)
    # Sprawdzenie, czy nowe wymiary nie są zerowe
    if new_w < 1 or new_h < 1:
        new_w = max(1, new_w)
        new_h = max(1, new_h)

    try:
        resample_filter = Image.Resampling.LANCZOS
    except AttributeError:
        resample_filter = Image.LANCZOS
    try:
        img = img.resize((new_w, new_h), resample_filter)
    except ValueError as e:
        print(f"Błąd resize w apply_scaling: {e} | Wymiary: ({new_w}, {new_h}) | Oryginalne: ({width}, {height})")
        return img, abs_boxes # Zwróć oryginał w razie błędu

    scale_x = img.size[0] / width if width > 0 else 0
    scale_y = img.size[1] / height if height > 0 else 0
    abs_boxes = [
        [cls, int(x1 * scale_x), int(y1 * scale_y), int(x2 * scale_x), int(y2 * scale_y)]
        for cls, x1, y1, x2, y2 in abs_boxes
    ]
    return img, abs_boxes

In [289]:
def apply_rotation(img):
    angle = random.uniform(-15, 15)
    img = img.convert("RGBA")
    try:
        # fillcolor jest poprawny dla RGBA
        img = img.rotate(angle, expand=True, resample=Image.Resampling.BILINEAR, fillcolor=(0, 0, 0, 0))
    except AttributeError: # Dla starszych wersji Pillow
        img = img.rotate(angle, expand=True, resample=Image.BILINEAR, fillcolor=(0, 0, 0, 0))
    return img

In [290]:
def apply_blur(img):
    if random.random() < 0.5:
        try:
            alpha = img.getchannel('A')
            rgb = img.convert('RGB').filter(ImageFilter.GaussianBlur(random.uniform(0, 1.5)))
            rgb.putalpha(alpha)
            img = rgb
        except (ValueError, IndexError): # Handle cases where getchannel might fail
             print("Ostrzeżenie: Nie udało się zastosować blur, problem z kanałem alfa?")
             pass # Kontynuuj z oryginalnym obrazem
    return img

In [291]:
def apply_brightness(img):
    if random.random() < 0.7:
        try:
            alpha = img.getchannel('A')
            rgb = img.convert('RGB')
            enhancer = ImageEnhance.Brightness(rgb)
            rgb = enhancer.enhance(random.uniform(0.6, 1.4))
            rgb.putalpha(alpha)
            img = rgb
        except (ValueError, IndexError):
            print("Ostrzeżenie: Nie udało się zastosować brightness, problem z kanałem alfa?")
            pass
    return img

In [292]:
def apply_contrast(img):
    if random.random() < 0.7:
        try:
            alpha = img.getchannel('A')
            rgb = img.convert('RGB')
            enhancer = ImageEnhance.Contrast(rgb)
            rgb = enhancer.enhance(random.uniform(0.6, 1.4))
            rgb.putalpha(alpha)
            img = rgb
        except (ValueError, IndexError):
            print("Ostrzeżenie: Nie udało się zastosować contrast, problem z kanałem alfa?")
            pass
    return img

In [293]:
def apply_color(img):
    if random.random() < 0.5:
        try:
            alpha = img.getchannel('A')
            rgb = img.convert('RGB')
            enhancer = ImageEnhance.Color(rgb)
            rgb = enhancer.enhance(random.uniform(0.6, 1.4))
            rgb.putalpha(alpha)
            img = rgb
        except (ValueError, IndexError):
            print("Ostrzeżenie: Nie udało się zastosować color, problem z kanałem alfa?")
            pass
    return img

In [294]:
def apply_flip(img, abs_boxes):
    if random.random() < 0.5:
        img = ImageOps.mirror(img)
        w = img.size[0]
        abs_boxes = [
            [cls, w - x2, y1, w - x1, y2] for cls, x1, y1, x2, y2 in abs_boxes
        ]
    return img, abs_boxes

In [295]:
def apply_noise(img):
    if random.random() < 0.5:
        try:
            np_img = np.array(img).astype(np.int16)
            # Sprawdzenie czy obraz ma kanał alfa
            if np_img.shape[2] == 4:
                rgb_channels = np_img[:, :, :3]
                alpha_channel = np_img[:, :, 3] # Zapisz kanał alfa
            else: # Jeśli obraz jest tylko RGB
                rgb_channels = np_img
                alpha_channel = None

            noise = np.random.normal(0, 15, rgb_channels.shape)
            rgb_channels = np.clip(rgb_channels + noise, 0, 255).astype(np.uint8)

            if alpha_channel is not None:
                # Odtwórz obraz RGBA
                final_img_np = np.dstack((rgb_channels, alpha_channel))
                img = Image.fromarray(final_img_np, 'RGBA')
            else:
                # Odtwórz obraz RGB
                img = Image.fromarray(rgb_channels, 'RGB')
        except Exception as e:
            print(f"Ostrzeżenie: Nie udało się zastosować noise: {e}")
            pass # Kontynuuj z oryginalnym obrazem
    return img

In [296]:
def apply_perspective_transform(img, abs_boxes):
    if random.random() < PERSPECTIVE_CHANCE:
        try:
            panel_w, panel_h = img.size
            if panel_w < 2 or panel_h < 2: return img, abs_boxes

            src_points = np.float32([[0, 0], [panel_w, 0], [panel_w, panel_h], [0, panel_h]])
            max_shift = max(panel_w, panel_h) * random.uniform(PERSPECTIVE_STRENGTH[0], PERSPECTIVE_STRENGTH[1])
            small_shift = max(panel_w, panel_h) * 0.05
            dst_points = np.float32([
                [random.uniform(0, max_shift), random.uniform(0, max_shift)],
                [panel_w - random.uniform(0, max_shift), random.uniform(0, max_shift)],
                [panel_w - random.uniform(0, small_shift), panel_h - random.uniform(0, small_shift)],
                [random.uniform(0, small_shift), panel_h - random.uniform(0, small_shift)]
            ])

            # Sprawdzenie, czy punkty nie tworzą zdegenerowanego czworokąta
            if np.linalg.det(np.array([[dst_points[0][0], dst_points[0][1], 1],
                                      [dst_points[1][0], dst_points[1][1], 1],
                                      [dst_points[2][0], dst_points[2][1], 1]])) == 0:
                print("Ostrzeżenie: Zdegenerowane punkty perspektywy, pomijam transformację.")
                return img, abs_boxes


            M = cv2.getPerspectiveTransform(src_points, dst_points)
            img_cv = np.array(img.convert('RGBA'))
            warped_img_cv = cv2.warpPerspective(img_cv, M, (panel_w, panel_h), borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
            warped_img = Image.fromarray(warped_img_cv, 'RGBA')

            bbox_pts = np.float32([[[0, 0]], [[panel_w, 0]], [[panel_w, panel_h]], [[0, panel_h]]])
            warped_bbox_pts = cv2.perspectiveTransform(bbox_pts, M)
            # Sprawdzenie czy transformacja nie zwróciła NaN
            if warped_bbox_pts is None or np.isnan(warped_bbox_pts).any():
                 print("Ostrzeżenie: Transformacja BBox zwróciła NaN, pomijam.")
                 return img, abs_boxes
            warped_bbox_pts = warped_bbox_pts.squeeze()


            x_min = np.min(warped_bbox_pts[:, 0])
            y_min = np.min(warped_bbox_pts[:, 1])
            x_max = np.max(warped_bbox_pts[:, 0])
            y_max = np.max(warped_bbox_pts[:, 1])

            # Dodatkowa walidacja koordynatów
            if x_max <= x_min or y_max <= y_min:
                print(f"Ostrzeżenie: Nieprawidłowy BBox po perspektywie ({x_min},{y_min},{x_max},{y_max}), pomijam.")
                return img, abs_boxes

            new_abs_boxes = [[CLASS_ID, int(x_min), int(y_min), int(x_max), int(y_max)]]
            return warped_img, new_abs_boxes

        except Exception as e:
            print(f"BŁĄD w apply_perspective_transform: {e}, pomijam transformację.")
            return img, abs_boxes # Zwróć oryginał w razie błędu

    return img, abs_boxes

In [297]:
def transform_image_and_boxes(img, abs_boxes):
    # Kolejność może mieć znaczenie - perspektywa na końcu?
    img, abs_boxes = apply_scaling(img, abs_boxes)
    img = apply_rotation(img)
    img = apply_blur(img)
    img = apply_brightness(img)
    img = apply_contrast(img)
    img = apply_color(img)
    img = apply_noise(img)
    img, abs_boxes = apply_flip(img, abs_boxes)
    # Zastosuj perspektywę na końcu, po innych zmianach rozmiaru/kształtu
    img, abs_boxes = apply_perspective_transform(img, abs_boxes)
    return img, abs_boxes

In [298]:
def get_bbox_for_image(img):
    w, h = img.size
    xc = 0.5
    yc = 0.5
    bw = 1.0
    bh = 1.0
    return [[CLASS_ID, xc, yc, bw, bh]]

def save_yolo_labels(label_path, boxes):
    with open(label_path, 'w') as f:
        for box in boxes:
            cls, xc, yc, w, h = box
            f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

def denormalize_boxes(boxes, width, height):
    abs_boxes = []
    for cls, xc, yc, w, h in boxes:
        abs_boxes.append([
            cls,
            int((xc - w / 2) * width),
            int((yc - h / 2) * height),
            int((xc + w / 2) * width),
            int((yc + h / 2) * height)
        ])
    return abs_boxes

def normalize_boxes(boxes, width, height):
    norm_boxes = []
    for cls, x1, y1, x2, y2 in boxes:
        x1 = max(0, min(x1, width))
        y1 = max(0, min(y1, height))
        x2 = max(0, min(x2, width))
        y2 = max(0, min(y2, height))
        xc = (x1 + x2) / 2 / width
        yc = (y1 + y2) / 2 / height
        w = (x2 - x1) / width
        h = (y2 - y1) / height
        if w > 0 and h > 0:
            norm_boxes.append([cls, xc, yc, w, h])
    return norm_boxes

Panele na dachach

In [299]:
def match_color_and_lighting(panel_img, background_patch):
    """Dopasowuje średnią jasność i kolorystykę panelu do fragmentu tła (dachu)."""
    try:
        panel_np = np.array(panel_img).astype(np.int16)
        alpha_channel = None
        if panel_np.shape[2] == 4:
            alpha_channel = panel_np[:, :, 3] # Zapisz alfę

        panel_rgb_img = panel_img.convert("RGB")
        background_stat = ImageStat.Stat(background_patch.convert("RGB"))
        panel_stat = ImageStat.Stat(panel_rgb_img)

        bg_mean = np.array(background_stat.mean)
        panel_mean = np.array(panel_stat.mean)
        mean_diff = bg_mean - panel_mean

        rgb_channels = panel_np[:,:,:3]
        for i in range(3):
            rgb_channels[:, :, i] = np.clip(rgb_channels[:, :, i] + mean_diff[i] * random.uniform(0.5, 1.5), 0, 255)

        corrected_rgb = rgb_channels.astype(np.uint8)

        if alpha_channel is not None:
            final_img_np = np.dstack((corrected_rgb, alpha_channel.astype(np.uint8)))
            return Image.fromarray(final_img_np, 'RGBA')
        else:
            return Image.fromarray(corrected_rgb, 'RGB')
    except Exception as e:
        print(f"Ostrzeżenie: Nie udało się dopasować kolorów: {e}")
        return panel_img # Zwróć oryginał w razie błędu

In [300]:
def composite_panel_realistically(panel_img_aug, abs_boxes_aug, background_img):
    """
    Nakłada panel na tło, kontrolując skalę, pozycję i dopasowanie kolorów.
    """
    try:
        bg_w, bg_h = background_img.size
        panel_w, panel_h = panel_img_aug.size

        if panel_w <=0 or panel_h <= 0: return None, None # Panel zniknął

        target_fill_percent = random.uniform(SCALE_RANGE[0], SCALE_RANGE[1])
        target_panel_w = int(bg_w * target_fill_percent)

        scale_needed = target_panel_w / panel_w if panel_w > 0 else 0
        if scale_needed == 0: return None, None

        new_w = target_panel_w
        new_h = int(panel_h * scale_needed)

        # Wymuszenie minimalnego rozmiaru 10x10 px
        new_w = max(MIN_PIXEL_SIZE, new_w)
        new_h = max(MIN_PIXEL_SIZE, new_h)

        panel_img_aug = panel_img_aug.resize((new_w, new_h), Image.Resampling.LANCZOS)
        panel_w, panel_h = new_w, new_h # Aktualizacja rozmiarów po resize

        abs_boxes_scaled = []
        for cls, x1, y1, x2, y2 in abs_boxes_aug:
            x1_s = int(x1 * scale_needed)
            y1_s = int(y1 * scale_needed)
            x2_s = int(x2 * scale_needed)
            y2_s = int(y2 * scale_needed)

            # Walidacja BBoxa po skalowaniu
            if x2_s <= x1_s or y2_s <= y1_s:
                print(f"Ostrzeżenie: Nieprawidłowy BBox po skalowaniu ({x1_s},{y1_s},{x2_s},{y2_s}), pomijam.")
                return None, None # BBox jest nieprawidłowy

            abs_boxes_scaled.append([cls, x1_s, y1_s, x2_s, y2_s])

        if not abs_boxes_scaled: # Jeśli lista BBoxów jest pusta
            return None, None

        # KONTROLA POZYCJI
        max_x = bg_w - panel_w
        max_y = int(bg_h * POSITION_MAX_Y) - panel_h

        if max_x < 0 or max_y < 0:
             # print(f"Ostrzeżenie: Panel {panel_w}x{panel_h} nie mieści się w dozwolonym obszarze {max_x}x{max_y}, pomijam.")
             return None, None

        x_offset = random.randint(0, max_x)
        y_offset = random.randint(0, max_y)

        # (Dopasowanie kolorów wyłączone)
        # background_patch = background_img.crop((x_offset, y_offset, x_offset + panel_w, y_offset + panel_h))
        # panel_img_aug = match_color_and_lighting(panel_img_aug, background_patch)

        # --- DIAGNOSTYKA ALFA PRZED PASTE ---
        try:
            alpha_check = panel_img_aug.getchannel('A')
            alpha_stat_check = ImageStat.Stat(alpha_check)
            if alpha_stat_check.sum[0] == 0:
                 print(f"OSTRZEŻENIE (Alfa przed Paste): Panel jest niewidzialny tuż przed wklejeniem!")
                 # Można tu zwrócić None, None, jeśli chcemy pominąć
                 # return None, None
            # else:
            #     print(f"    DEBUG (Alfa przed Paste): Alfa OK, suma={alpha_stat_check.sum[0]}")
        except Exception as e_alpha_paste:
            print(f"Ostrzeżenie: Błąd sprawdzania alfy przed paste: {e_alpha_paste}")
            # Kontynuujmy mimo błędu sprawdzania

        # KOMPOZYCJA
        composite_img = background_img.copy()
        composite_img.paste(panel_img_aug, (x_offset, y_offset), panel_img_aug)

        # AKTUALIZACJA BBOXA
        final_abs_boxes = []
        for cls, x1, y1, x2, y2 in abs_boxes_scaled:
            fx1 = x1 + x_offset
            fy1 = y1 + y_offset
            fx2 = x2 + x_offset
            fy2 = y2 + y_offset

            # OSTATECZNA WALIDACJA BBOXA PRZED ZWROCENIEM
            if fx2 <= fx1 or fy2 <= fy1:
                 print(f"Ostrzeżenie: Finalny BBox nieprawidłowy ({fx1},{fy1},{fx2},{fy2}), pomijam.")
                 return None, None

            final_abs_boxes.append([cls, fx1, fy1, fx2, fy2])

        if not final_abs_boxes: # Jeśli lista jest pusta po walidacji
             return None, None

        return composite_img, final_abs_boxes

    except Exception as e:
        print(f"BŁĄD w composite_panel_realistically: {e}")
        return None, None # Zwróć None w razie jakiegokolwiek błędu w tej funkcji

In [301]:
def get_initial_bbox(img):
    return [[CLASS_ID, 0.5, 0.5, 1.0, 1.0]]

def denormalize_boxes(boxes, width, height):
    abs_boxes = []
    for cls, xc, yc, w, h in boxes:
        abs_boxes.append([
            cls,
            int((xc - w / 2) * width),
            int((yc - h / 2) * height),
            int((xc + w / 2) * width),
            int((yc + w / 2) * height)
        ])
    return abs_boxes

def normalize_boxes(boxes, width, height):
    norm_boxes = []
    if width <= 0 or height <= 0: # Zabezpieczenie przed dzieleniem przez zero
        return []

    for cls, x1, y1, x2, y2 in boxes:
        # Podstawowa walidacja koordynatów wejściowych
        if x2 <= x1 or y2 <= y1:
            print(f"Ostrzeżenie w normalize_boxes: Otrzymano nieprawidłowy box wejściowy [{x1},{y1},{x2},{y2}], pomijam.")
            continue

        x1 = max(0, min(x1, width))
        y1 = max(0, min(y1, height))
        x2 = max(0, min(x2, width))
        y2 = max(0, min(y2, height))

        # Ponowna walidacja po przycięciu do granic
        if x2 <= x1 or y2 <= y1:
            # print(f"Ostrzeżenie w normalize_boxes: Box stał się nieprawidłowy po przycięciu [{x1},{y1},{x2},{y2}], pomijam.")
            continue

        xc = (x1 + x2) / 2 / width
        yc = (y1 + y2) / 2 / height
        w = (x2 - x1) / width
        h = (y2 - y1) / height

        # Ostateczna walidacja znormalizowanych wymiarów
        if w > 0 and h > 0:
            norm_boxes.append([cls, xc, yc, w, h])
        # else:
        #     print(f"Ostrzeżenie w normalize_boxes: Znormalizowane w={w} lub h={h} jest <= 0, pomijam.")

    return norm_boxes

def save_yolo_labels(label_path, boxes):
    """Zapisuje znormalizowane boxy do pliku .txt w formacie YOLO"""
    with open(label_path, 'w') as f:
        for box in boxes:
            cls, xc, yc, w, h = box
            f.write(f"{int(cls)} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

Pętla główna

In [302]:
# for file in os.listdir(PANEL_DIR):
#     if not file.lower().endswith(('.jpg', '.png')):
#         continue

#     img_path = os.path.join(PANEL_DIR, file)
#     original_img = Image.open(img_path)
#     boxes = get_bbox_for_image(original_img)

#     for i in range(AUG_PER_IMAGE):
#         img = original_img.copy()
#         abs_boxes = denormalize_boxes(boxes, img.width, img.height)
#         aug_img, aug_boxes = transform_image_and_boxes(img, abs_boxes)
#         norm_boxes = normalize_boxes(aug_boxes, aug_img.width, aug_img.height)

#         out_name = f"{os.path.splitext(file)[0]}_aug_{i:03d}"
#         aug_img.save(os.path.join(OUTPUT_IMG_DIR, out_name + ".jpg"))
#         save_yolo_labels(os.path.join(OUTPUT_LABEL_DIR, out_name + ".txt"), norm_boxes)

#     print(f"✔ {file} - {AUG_PER_IMAGE} augmentacji")

In [303]:
print("Rozpoczynam generowanie danych...")
start_time_total = time.time()

# 1. Wczytanie listy plików tła (dachów)
roof_files = [f for f in os.listdir(ROOF_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
if not roof_files:
    print(f"BŁĄD KRYTYCZNY: Brak plików w katalogu dachów: {ROOF_DIR}. Przerwij i dodaj obrazy tła.")
else:
    print(f"Znaleziono {len(roof_files)} obrazów tła.")

# 2. Pętla po plikach paneli
panel_files = [f for f in os.listdir(PANEL_DIR) if f.lower().endswith('.png')]
if not panel_files:
    print(f"BŁĄD KRYTYCZNY: Brak plików .png w katalogu paneli: {PANEL_DIR}. Przerwij i dodaj wycięte panele.")
else:
    print(f"Znaleziono {len(panel_files)} plików paneli. Rozpoczynam pętlę główną...")

total_generated_count = 0
successful_counts = {} # Licznik udanych generacji dla każdego panelu

for panel_file in panel_files:
    successful_counts[panel_file] = 0 # Inicjalizacja licznika
    if not roof_files:
        break

    print(f"\n--- Przetwarzanie panelu: {panel_file} ---")
    start_time_panel = time.time()

    panel_path = os.path.join(PANEL_DIR, panel_file)
    try:
        original_panel_img = Image.open(panel_path).convert("RGBA")
    except Exception as e:
        print(f"BŁĄD KRYTYCZNY: Nie można otworzyć pliku panelu {panel_file}: {e}. Pomijam ten panel.")
        continue # Przejdź do następnego panelu

    initial_boxes_norm = get_initial_bbox(original_panel_img)
    base_name = os.path.splitext(panel_file)[0]

    attempt_count = 0 # Licznik prób dla tego panelu
    generated_count = 0 # Licznik udanych generacji dla tego panelu
    max_attempts = AUG_PER_COMPOSITE * 5 # Zwiększamy liczbę prób, jeśli wiele kończy się niepowodzeniem

    while generated_count < AUG_PER_COMPOSITE and attempt_count < max_attempts:
        attempt_count += 1
        try:
            # 1. Wczytanie tła
            roof_file = random.choice(roof_files)
            roof_path = os.path.join(ROOF_DIR, roof_file)
            try:
                roof_img = Image.open(roof_path).convert("RGB")
            except Exception as e_open_roof:
                 print(f"Ostrzeżenie: Nie można otworzyć pliku tła {roof_file}: {e_open_roof}. Próbuję następne.")
                 continue # Spróbuj inne tło


            # --- Normalizacja Tła ---
            try:
                resample_filter = Image.Resampling.LANCZOS
            except AttributeError:
                resample_filter = Image.LANCZOS
            roof_img = roof_img.resize(BACKGROUND_TARGET_SIZE, resample_filter)

            # 2. Augmentacja panelu
            panel_copy = original_panel_img.copy()
            abs_boxes = denormalize_boxes(initial_boxes_norm, panel_copy.width, panel_copy.height)
            aug_panel, aug_abs_boxes = transform_image_and_boxes(panel_copy, abs_boxes)

            # Sprawdzenie czy panel nie zniknął po augmentacji
            if aug_panel is None or aug_panel.size[0] < 1 or aug_panel.size[1] < 1:
                # print(f"    DEBUG: Panel zniknął po augmentacji.")
                continue

            # 3. Realistyczna Kompozycja
            composite_img, final_abs_boxes = composite_panel_realistically(
                aug_panel,
                aug_abs_boxes,
                roof_img
            )

            if composite_img is None:
                # print(f"    DEBUG: Kompozycja zwróciła None.")
                continue # Pominięcie (np. panel za mały, nie mieści się, BBox nieprawidłowy)

            # 4. Normalizacja i zapis
            norm_boxes = normalize_boxes(final_abs_boxes, composite_img.width, composite_img.height)

            if not norm_boxes:
                # print(f"    DEBUG: Normalizacja zwróciła pustą listę.")
                continue # Pominięcie (BBox stał się nieprawidłowy po normalizacji/przycięciu)

            out_name = f"{base_name}_roof_{generated_count:03d}"
            img_save_path = os.path.join(OUTPUT_IMG_DIR, out_name + ".jpg")
            label_save_path = os.path.join(OUTPUT_LABEL_DIR, out_name + ".txt")

            composite_img.save(img_save_path)
            save_yolo_labels(label_save_path, norm_boxes)

            generated_count += 1
            total_generated_count += 1
            successful_counts[panel_file] = generated_count # Aktualizacja licznika udanych

            if generated_count % 10 == 0:
                print(f"    ...wygenerowano {generated_count}/{AUG_PER_COMPOSITE} obrazów dla {panel_file} (próba {attempt_count})")

        except Exception as e:
            print(f"BŁĄD (w głównym bloku try): {e} dla {panel_file} (tło: {roof_file})")
            # Nie przerywamy pętli, próbujemy dalej
            continue

    panel_time = time.time() - start_time_panel
    if generated_count < AUG_PER_COMPOSITE:
         print(f"⚠️ Panel {panel_file}: Zakończono przedwcześnie po {attempt_count} próbach. Zapisano tylko {generated_count}/{AUG_PER_COMPOSITE} obrazów w {panel_time:.2f}s.")
    else:
         print(f"✔ Panel {panel_file}: Zakończono. Zapisano {generated_count} obrazów w {panel_time:.2f}s.")

total_time = time.time() - start_time_total
print(f"\n✅ Generowanie zakończone.")
print("\nPodsumowanie wygenerowanych obrazów:")
for panel, count in successful_counts.items():
    print(f"- {panel}: {count}")
print(f"\nŁącznie wygenerowano {total_generated_count} obrazów w {total_time:.2f}s.")
print(f"Zbiór danych znajduje się w '{OUTPUT_BASE_DIR}'.")

Rozpoczynam generowanie danych...
Znaleziono 30 obrazów tła.
Znaleziono 5 plików paneli. Rozpoczynam pętlę główną...

--- Przetwarzanie panelu: 1402181-small-removebg-preview.png ---
OSTRZEŻENIE (Alfa przed Paste): Panel jest niewidzialny tuż przed wklejeniem!
OSTRZEŻENIE (Alfa przed Paste): Panel jest niewidzialny tuż przed wklejeniem!
OSTRZEŻENIE (Alfa przed Paste): Panel jest niewidzialny tuż przed wklejeniem!
OSTRZEŻENIE (Alfa przed Paste): Panel jest niewidzialny tuż przed wklejeniem!
OSTRZEŻENIE (Alfa przed Paste): Panel jest niewidzialny tuż przed wklejeniem!
OSTRZEŻENIE (Alfa przed Paste): Panel jest niewidzialny tuż przed wklejeniem!
    ...wygenerowano 10/50 obrazów dla 1402181-small-removebg-preview.png (próba 10)
OSTRZEŻENIE (Alfa przed Paste): Panel jest niewidzialny tuż przed wklejeniem!
OSTRZEŻENIE (Alfa przed Paste): Panel jest niewidzialny tuż przed wklejeniem!
OSTRZEŻENIE (Alfa przed Paste): Panel jest niewidzialny tuż przed wklejeniem!
OSTRZEŻENIE (Alfa przed Paste):